# MINATO: binary population synthesis and survey simulation

This notebook is a practical guide to the `minato.binary_population` subpackage. It focuses on:

- how the **intrinsic population** is drawn (`BinaryPopulation`)
- how **survey cadence and RV errors** are applied (`BinarySurveySimulator`)
- what the key **safety/physics guards** do (Roche-limit guard, exposure-smear flag, eccentricity caps)
- how to run the included **binary-fraction inference** helper (`run_mcmc`)

See also: `minato/binary_population/README.md`.


## Prerequisites

This tutorial assumes you are running in the MINATO conda environment (`minato_env.yml`) and that optional dependencies for `minato.binary_population` are available (notably `kepler`, and for `run_mcmc` also `emcee`).

"If `import minato.binary_population` fails, we first make sure the notebook can find the local MINATO source tree (i.e. the `minato/` directory)


In [ ]:
import sys
from pathlib import Path

try:
    import minato  # noqa: F401
except ModuleNotFoundError:
    here = Path.cwd().resolve()
    for root in [here, *here.parents]:
        if (root / "minato" / "__init__.py").exists():
            sys.path.insert(0, str(root))
            break
    else:
        raise RuntimeError(
            "Could not locate the MINATO repository root (missing 'minato/__init__.py' in parents of cwd). "
            "Launch Jupyter from the MINATO repo root, or install MINATO into your environment."
        )


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from minato.binary_population import BinaryPopulation, BinarySurveySimulator

np.random.seed(42)
plt.rcParams.update({"figure.figsize": (7, 4)})


## 1) Intrinsic populations with `BinaryPopulation`

`BinaryPopulation` draws an *intrinsic* catalogue of stars, where a fraction `f_bin` are binaries.
For binaries it samples:

- primary mass `M1`
- mass ratio `q`
- period `P` (via `logP`)
- eccentricity `e` (optionally capped as a function of `P`)
- orbital angles and systemic velocity

The output is a pandas `DataFrame` with one row per synthetic system.


In [ ]:
pop = BinaryPopulation()
pop.summary()

### Key safety / realism toggles

These are the most important switches to understand when results look surprising:

- `use_period_ecc_cap`: enforce an eccentricity cap that depends on period (common in massive-binary population modelling)
- `e_max`: absolute hard ceiling on eccentricity
- `use_roche_guard`: clamps too-short periods upward to avoid Roche-lobe overflow (at periastron) by a margin `roche_margin_frac`
- `use_smear_flag`: computes an exposure-smear flag for a nominal exposure time `t_exp_sec` and threshold `dv_smear_limit`


In [ ]:
# Example: make the guard settings explicit (defaults shown here)
pop.use_period_ecc_cap = True
pop.e_max = 0.95

pop.use_roche_guard = True
pop.roche_margin_frac = 0.10

pop.use_smear_flag = True
pop.t_exp_sec = 900.0
pop.dv_smear_limit = 20.0

pop.summary()

### Draw a small intrinsic sample

Tip: keep `N` small while experimenting, then scale up once you are happy with the configuration.


In [ ]:
intrinsic = pop.generate_intrinsic_sample_vectorized(N=300, f_bin=0.7)
intrinsic.head()

In [ ]:
# How often did the Roche guard clamp the drawn period?
if "P_clamped" in intrinsic.columns:
    frac = intrinsic.loc[intrinsic["is_binary"], "P_clamped"].mean()
    print(f"Fraction of binaries with Roche-clamped periods: {frac:.3f}")

# Visualize basic distributions for binaries only
b = intrinsic[intrinsic["is_binary"]].copy()

fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
ax[0].hist(b["M1"], bins=20)
ax[0].set_xlabel(r"$M_1\,(M_\odot)$")
ax[0].set_ylabel("count")

ax[1].hist(np.log10(b["P"]), bins=20)
ax[1].set_xlabel(r"$\log_{10} P\,(\mathrm{days})$")

ax[2].hist(b["e"], bins=20)
ax[2].set_xlabel(r"$e$")

fig.tight_layout()

## 2) Fixed-value overrides ("grid-like" experiments)

If you want controlled experiments (e.g. fixed `M1`, `q`, `P`, `e` lists), you can set:

- `M1_values`, `q_values`, `logP_values`, `e_values`
- `fixed_values_mode = "random"` (default) or `"cycle"`
- optional `fixed_values_weights` (for random mode)

When fixed `e` values violate the eccentricity caps, `fixed_e_enforcement` controls behavior:

- `"clip"`: clip to the cap
- `"error"`: raise an error


In [ ]:
pop2 = BinaryPopulation()

# Fix some parameters to a small list
pop2.M1_values = [10.0, 15.0, 20.0]
pop2.q_values = [0.2, 0.5, 1.0]
pop2.logP_values = list(np.log10([2.0, 5.0, 20.0]))
pop2.e_values = [0.0, 0.3, 0.8]

# Choose how to draw from fixed lists
pop2.fixed_values_mode = "cycle"  # try also: "random"
pop2.fixed_e_enforcement = "clip"  # try also: "error"

intrinsic2 = pop2.generate_intrinsic_sample_vectorized(N=60, f_bin=1.0)
intrinsic2[["M1", "q", "P", "e"]].head(10)

## 3) Survey simulation with `BinarySurveySimulator`

`BinarySurveySimulator` converts an intrinsic catalogue into *observed* RV time series by specifying:

- a cadence and RV error model (real coverage or synthetic)
- an observing strategy (`ideal_sampling`)

Common modes:

- `ideal_sampling=True`: 2 epochs at RV extrema ("two quadratures")
- `ideal_sampling="phase_uniform"`: N epochs uniformly in orbital phase
- `ideal_sampling=False`: uses a real cadence loaded via `load_data()`


In [ ]:
survey = BinarySurveySimulator(pop)

# Ideal sampling modes (two quadratures / phase-uniform) apply only to binaries.
# Filter to binaries (or generate an intrinsic sample with f_bin=1.0).
intrinsic_bin = intrinsic[intrinsic["is_binary"]].copy()

# Two-quadrature observations with a fixed RV uncertainty
obs_2q = survey.simulate_mock_observations(
    intrinsic_sample=intrinsic_bin,
    ideal_sampling=True,
    rv_error_common=2.0,
    seed=123,
)
obs_2q.head()


In [ ]:
# Example detection metrics:
# - dRV_max: max RV span from noisy measurements
# - sigma_d: detection significance proxy (max pairwise |Δv| / σ_Δv)
bobs = obs_2q[obs_2q["is_binary"]].copy()

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].hist(bobs["dRV_max"], bins=30)
ax[0].set_xlabel(r"$\Delta RV_{\max}$ (km/s)")
ax[0].set_ylabel("count")

ax[1].hist(bobs["sigma_d"], bins=30)
ax[1].set_xlabel(r"$\sigma_d$ (max pairwise significance)")
fig.tight_layout()

In [ ]:
# Uniform-in-phase sampling (N epochs). This requires rv_error_common because this mode does not use a real coverage/error distribution.
obs_phase = survey.simulate_mock_observations(
    intrinsic_sample=intrinsic_bin,
    ideal_sampling="phase_uniform",
    n_epochs=8,
    rv_error_common=2.0,
    seed=123,
)
obs_phase[["synthetic_ID", "is_binary", "n_eps", "dRV_max", "sigma_d"]].head()


### Using real cadence

To use real cadence, you must provide a DataFrame with columns:

- `ID`: identifier for each real star
- `MJD`: observation time
- `mean_rv_er`: per-epoch RV uncertainty

Below is a *toy* example that mimics this structure.


In [ ]:
# Toy cadence for two "real" stars (replace with your survey coverage table)
coverage_df = pd.DataFrame(
    {
        "ID": ["starA"] * 6 + ["starB"] * 6,
        "MJD": np.r_[
            59000 + np.sort(np.random.uniform(0, 80, 6)),
            59000 + np.sort(np.random.uniform(0, 80, 6)),
        ],
        "mean_rv_er": np.r_[np.full(6, 3.0), np.full(6, 5.0)],
    }
)

survey_real = BinarySurveySimulator(pop)
survey_real.load_data(coverage_df)

obs_real = survey_real.simulate_mock_observations(
    N=200,
    f_bin=0.6,
    ideal_sampling=False,
    seed=7,
)

display_cols = ["real_ID_used", "dRV_max", "sigma_d"]
# Column naming note: older outputs used `n_eps`; newer outputs include `n_epochs` (alias).
if "n_epochs" in obs_real.columns:
    display_cols.insert(1, "n_epochs")
elif "n_eps" in obs_real.columns:
    display_cols.insert(1, "n_eps")
obs_real[display_cols].head()
# obs_real.head()


## 4) Inferring the binary fraction with `run_mcmc`

`run_mcmc` fits the binary fraction `f_bin` by comparing the distribution of `\Delta RV_{\max}` between data and simulations with a Poisson likelihood.

### Why this can be slow

The likelihood is *simulation-based*: for each proposed `f_bin`, it generates a mock sample of size `N_sim` (in batches) and builds a histogram of `\Delta RV_{\max}`. In MCMC this likelihood is evaluated many times (walkers × steps), so runtime grows quickly with `N_sim`, `nwalkers`, and `nsteps`.

### Tutorial best practices

- Start with small values (order `N_sim=1000--3000`, `nwalkers=8--12`, `nsteps=200--400`).
- Set `batch_size=N_sim` for the demo (single batch; less overhead).
- Increase `N_sim` first (reduces Monte-Carlo noise), then increase `nsteps` if needed.
- Use a real cadence table (`ideal_sampling=False` with `survey.load_data(...)`) so single stars are supported.

### Parallelism: `pool_kind` and `nthreads`

- `pool_kind="none"`: no parallelism (simplest, good for quick smoke tests).
- `pool_kind="thread"`: uses a thread pool; may help a little, but can be limited by the GIL for Python-heavy likelihoods.
- `pool_kind="process"` (recommended for multi-core): uses multiprocessing and typically scales better on large CPUs.

Tips:
- Start with `pool_kind="none"` to confirm everything runs.
- On macOS, use `start_method="spawn"` when using `pool_kind="process"`.
- Choose `nthreads` based on your machine: 2–8 on a laptop, and higher on servers; speedups plateau when overhead dominates.
- To keep many cores busy, you generally need enough walkers (`nwalkers` comparable to or larger than `nthreads`).


In [ ]:
from minato.binary_population import run_mcmc

# Example "observed" dRV_max (here: reuse a simulated sample as a stand-in).
# In real use, this should come from your survey RV measurements.
dRV_real = obs_real["dRV_max"].to_numpy()

RUN_MCMC = True

# Fast demo settings (intended to run in ~tens of seconds to a couple of minutes).
# Increase N_sim / nsteps for more stable posteriors.
N_sim = 2000
batch_size = N_sim
nwalkers = 10
nsteps = 300
nthreads = 8
pool_kind = "process" # try also: "thread", "none"
start_method = "spawn" # try "fork" if on Linux

if RUN_MCMC:
    sampler = run_mcmc(
        pop,
        survey_real,
        dRV_real=dRV_real,
        N_sim=N_sim,
        batch_size=batch_size,
        nwalkers=nwalkers,
        nsteps=nsteps,
        nthreads=nthreads,
        pool_kind=pool_kind,
        start_method=start_method,
        sim_kwargs={"ideal_sampling": False},
    )
else:
    sampler = None
    print("Set RUN_MCMC=True to run the demo (requires emcee).")


In [ ]:
# Plot MCMC results (trace + posterior)
if sampler is not None:
    chain = sampler.get_chain()[:, :, 0]  # (nsteps, nwalkers)
    burn = max(0, int(0.3 * chain.shape[0]))
    flat = sampler.get_chain(discard=burn, thin=5, flat=True)[:, 0]

    print("acceptance fraction (mean)", np.mean(sampler.acceptance_fraction))
    print("f_bin median", float(np.median(flat)))
    print("f_bin 16/84", tuple(np.percentile(flat, [16, 84])))

    fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=False)
    ax[0].plot(chain, alpha=0.35, linewidth=0.7)
    ax[0].axvline(burn, color="k", linestyle=":", linewidth=1)
    ax[0].set_ylabel(r"$f_{\rm bin}$")
    ax[0].set_title("run_mcmc trace (burn-in marked)")

    ax[1].hist(flat, bins=30, density=True, alpha=0.8)
    ax[1].set_xlabel(r"$f_{\rm bin}$")
    ax[1].set_ylabel("posterior density")
    fig.tight_layout()

    # Optional: save figure for the paper/notes
    # fig.savefig("mcmc_fbin_demo.png", dpi=200, bbox_inches="tight")


## Notes and common gotchas

- If many periods are being clamped, inspect `use_roche_guard`, `roche_margin_frac`, and your mass/radius assumptions (`radius_source`).
- For `ideal_sampling=True` (two quadratures), set `rv_error_common` unless you load a real coverage dataset.
- For `ideal_sampling="phase_uniform"`, you must set both `n_epochs` and `rv_error_common`.
- `minato.binary_simulator` is deprecated; prefer `minato.binary_population`.
